# Sleep Disorder model rebuild

Compact, product-oriented notebook for the `sleep_disorder` target.

Goals:
- keep feature sets short enough for a quiz
- always include `age_years`, `gender`, `med_count`
- test whether standard Check-up-35 style labs help
- use tuned Logistic Regression with scaling
- compare compact runs fairly on the same train/test split

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import re
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

In [ ]:
DATA_PATH = Path("../data/processed/nhanes_merged_adults_final.csv")
OUTPUT_DIR = Path("../data/processed/model_outputs_sleep_disorder_rebuild")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "sleep_disorder"
RANDOM_STATE = 42
TEST_SIZE = 0.20

print("DATA_PATH:", DATA_PATH.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Target present:", TARGET in df.columns)
print("First 12 columns:", df.columns[:12].tolist())

In [ ]:
assert TARGET in df.columns, f"{TARGET} not found in dataframe."

print(df[TARGET].value_counts(dropna=False).sort_index())
print()
print("Target prevalence:")
print(df[TARGET].value_counts(normalize=True, dropna=False).sort_index().round(4))

## Clean questionnaire-style special codes

NHANES often uses 7 / 9 / 77 / 99 style values for "refused", "don't know", etc.
We convert common questionnaire-like columns to `NaN` before modeling.

In [ ]:
SPECIAL_MISSING_CODES = {
    7, 9,
    77, 99,
    777, 999,
    7777, 9999,
    77777, 99999,
}

QUESTIONNAIRE_LIKE_PREFIXES = (
    "slq", "sld", "dpq", "paq", "huq", "mcq", "bpq", "kiq", "cdq", "smq", "alq", "whq"
)

def replace_special_codes_with_nan(series: pd.Series) -> pd.Series:
    if not pd.api.types.is_numeric_dtype(series):
        return series
    return series.replace(list(SPECIAL_MISSING_CODES), np.nan)

questionnaire_like_cols = [
    c for c in df.columns
    if c.lower().startswith(QUESTIONNAIRE_LIKE_PREFIXES)
]

print("Questionnaire-like columns to clean:", len(questionnaire_like_cols))
df[questionnaire_like_cols] = df[questionnaire_like_cols].apply(replace_special_codes_with_nan)

In [ ]:
overview = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_pct": (df.isna().mean() * 100).round(2).values,
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
}).sort_values(["missing_pct", "n_unique"], ascending=[False, True])

overview.head(30)

## Compact feature design

We deliberately keep the product-facing feature sets short.
For sleep disorder, standard Check-up-35 labs are not expected to be as strong as in
metabolically-driven conditions, but we test them as optional context.

In [ ]:
DEMO_COLS = [
    "age_years",
    "gender",
]

MED_COLS = [
    "med_count",
]

SCREENING_LAB_COLS = [
    "fasting_glucose_mg_dl",
    "hdl_cholesterol_mg_dl",
    "total_cholesterol_mg_dl",
    "triglycerides_mg_dl",
    "uacr_mg_g",
]

# Optional richer context labs if available. These are not necessarily standard Check-up-35 uploads.
AUGMENTED_LAB_COLS = [
    "serum_creatinine_mg_dl",
    "bmi",
]

SLEEP_QUIZ_COLS = [
    "slq030___how_often_do_you_snore?",
    "sld012___sleep_hours___weekdays_or_workdays",
    "sld013___sleep_hours___weekends",
    "dpq040___feeling_tired_or_having_little_energy",
    "bpq020___ever_told_you_had_high_blood_pressure",
    "cdq010___shortness_of_breath_on_stairs/inclines",
    "paq665___moderate_recreational_activities",
    "alq111___ever_had_a_drink_of_any_kind_of_alcohol",
    "mcq010___ever_been_told_you_have_asthma",
]

def existing_cols(cols, frame):
    return [c for c in cols if c in frame.columns]

def missing_cols(cols, frame):
    return [c for c in cols if c not in frame.columns]

def dedupe_keep_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

print("Available sleep quiz cols:", existing_cols(SLEEP_QUIZ_COLS, df))
print("Missing sleep quiz cols:", missing_cols(SLEEP_QUIZ_COLS, df))

print("\nAvailable demo cols:", existing_cols(DEMO_COLS, df))
print("Available med cols:", existing_cols(MED_COLS, df))
print("Available screening labs:", existing_cols(SCREENING_LAB_COLS, df))
print("Available augmented labs:", existing_cols(AUGMENTED_LAB_COLS, df))

In [ ]:
feature_sets = {
    "compact_quiz_only": dedupe_keep_order(
        existing_cols(SLEEP_QUIZ_COLS, df)
    ),
    "compact_quiz_demo_med": dedupe_keep_order(
        existing_cols(SLEEP_QUIZ_COLS, df)
        + existing_cols(DEMO_COLS, df)
        + existing_cols(MED_COLS, df)
    ),
    "compact_quiz_demo_med_screening_labs": dedupe_keep_order(
        existing_cols(SLEEP_QUIZ_COLS, df)
        + existing_cols(DEMO_COLS, df)
        + existing_cols(MED_COLS, df)
        + existing_cols(SCREENING_LAB_COLS, df)
    ),
    "compact_quiz_demo_med_augmented_labs": dedupe_keep_order(
        existing_cols(SLEEP_QUIZ_COLS, df)
        + existing_cols(DEMO_COLS, df)
        + existing_cols(MED_COLS, df)
        + existing_cols(SCREENING_LAB_COLS, df)
        + existing_cols(AUGMENTED_LAB_COLS, df)
    ),
}

pd.DataFrame({
    "run": list(feature_sets.keys()),
    "n_features": [len(v) for v in feature_sets.values()],
    "features": [", ".join(v) for v in feature_sets.values()],
})

## Shared split for fair comparisons

In [ ]:
all_model_features = dedupe_keep_order(sum(feature_sets.values(), []))

model_df = df[[TARGET] + all_model_features].copy()
mask = model_df[TARGET].notna()

X_all = model_df.loc[mask, all_model_features].copy()
y_all = model_df.loc[mask, TARGET].astype(int).copy()

X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=TEST_SIZE,
    stratify=y_all,
    random_state=RANDOM_STATE,
)

print("Train shape:", X_train_all.shape)
print("Test shape:", X_test_all.shape)
print("Train target distribution:")
print(y_train.value_counts(normalize=True).round(4))

In [ ]:
def infer_feature_type(series: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(series):
        nunique = series.nunique(dropna=True)
        if nunique <= 10:
            return "categorical"
        return "numeric"
    return "categorical"

def split_feature_types(X_frame: pd.DataFrame, feature_list):
    numeric_features = []
    categorical_features = []
    for col in feature_list:
        if infer_feature_type(X_frame[col]) == "numeric":
            numeric_features.append(col)
        else:
            categorical_features.append(col)
    return numeric_features, categorical_features

def build_logreg_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop"
    )

def evaluate_predictions(y_true, y_proba, threshold=0.50):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "avg_precision": average_precision_score(y_true, y_proba),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "classification_report": classification_report(y_true, y_pred, zero_division=0),
    }

def run_tuned_logreg(feature_list, run_name):
    X_train = X_train_all[feature_list].copy()
    X_test = X_test_all[feature_list].copy()

    numeric_features, categorical_features = split_feature_types(X_train, feature_list)
    preprocessor = build_logreg_preprocessor(numeric_features, categorical_features)

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])

    param_grid = [
        {
            "model__solver": ["liblinear"],
            "model__penalty": ["l1", "l2"],
            "model__C": [0.01, 0.1, 1, 3, 10],
        },
        {
            "model__solver": ["saga"],
            "model__penalty": ["l1", "l2"],
            "model__C": [0.01, 0.1, 1, 3, 10],
        },
    ]

    cv = RepeatedStratifiedKFold(
        n_splits=5,
        n_repeats=2,
        random_state=RANDOM_STATE
    )

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="average_precision",
        cv=cv,
        n_jobs=-1,
        verbose=1,
        refit=True
    )

    grid.fit(X_train, y_train)

    best_estimator = grid.best_estimator_
    y_test_proba = best_estimator.predict_proba(X_test)[:, 1]

    metrics = evaluate_predictions(y_test, y_test_proba, threshold=0.50)

    return {
        "run": run_name,
        "feature_list": feature_list,
        "n_features": len(feature_list),
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "best_estimator": best_estimator,
        "best_params": grid.best_params_,
        "cv_best_score_avg_precision": grid.best_score_,
        "y_test_true": y_test.copy(),
        "y_test_proba": y_test_proba,
        **metrics,
    }

In [ ]:
all_results = {}

for run_name, feature_list in feature_sets.items():
    print(f"\n=== Running {run_name} ===")
    all_results[run_name] = run_tuned_logreg(feature_list, run_name)

In [ ]:
comparison_df = pd.DataFrame([
    {
        "run": run_name,
        "n_features": result["n_features"],
        "cv_best_score_avg_precision": round(result["cv_best_score_avg_precision"], 4),
        "roc_auc": round(result["roc_auc"], 4),
        "avg_precision": round(result["avg_precision"], 4),
        "precision": round(result["precision"], 4),
        "recall": round(result["recall"], 4),
        "f1": round(result["f1"], 4),
        "best_params": json.dumps(result["best_params"]),
    }
    for run_name, result in all_results.items()
]).sort_values(["avg_precision", "roc_auc"], ascending=False).reset_index(drop=True)

comparison_df

## Choose preferred product model

Metrics may favor one run, but the best product model can still be a different one.
Default below is the screening-lab variant because it stays close to a standard lab upload workflow.

In [ ]:
auto_best_run = comparison_df.iloc[0]["run"]
auto_best_result = all_results[auto_best_run]

selected_run = "compact_quiz_demo_med_screening_labs"
selected_result = all_results[selected_run]

print("Auto-best run:", auto_best_run)
print("Selected run:", selected_run)
print("Best params:", selected_result["best_params"])
print("ROC-AUC:", round(selected_result["roc_auc"], 4))
print("Average Precision:", round(selected_result["avg_precision"], 4))
print("Precision:", round(selected_result["precision"], 4))
print("Recall:", round(selected_result["recall"], 4))
print("F1:", round(selected_result["f1"], 4))
print()
print("Confusion Matrix:")
print(selected_result["confusion_matrix"])
print()
print(selected_result["classification_report"])

## Threshold scan

In [ ]:
threshold_rows = []
y_true = selected_result["y_test_true"]
y_proba = selected_result["y_test_proba"]

roc_auc = roc_auc_score(y_true, y_proba)

for thr in np.arange(0.10, 0.91, 0.05):
    metrics = evaluate_predictions(y_true, y_proba, threshold=float(thr))
    threshold_rows.append({
        "threshold": round(float(thr), 2),
        "roc_auc": round(roc_auc, 4),
        "precision": round(metrics["precision"], 4),
        "recall": round(metrics["recall"], 4),
        "f1": round(metrics["f1"], 4),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df

## Coefficients in a readable form

In [ ]:
FEATURE_LABEL_MAP = {
    "age_years": "Age (years)",
    "gender": "Gender",
    "med_count": "Medication count",
    "fasting_glucose_mg_dl": "Fasting glucose",
    "hdl_cholesterol_mg_dl": "HDL cholesterol",
    "total_cholesterol_mg_dl": "Total cholesterol",
    "triglycerides_mg_dl": "Triglycerides",
    "uacr_mg_g": "Urine albumin-creatinine ratio (UACR)",
    "serum_creatinine_mg_dl": "Serum creatinine",
    "bmi": "BMI",
    "slq030___how_often_do_you_snore?": "How often do you snore?",
    "sld012___sleep_hours___weekdays_or_workdays": "Sleep hours on weekdays",
    "sld013___sleep_hours___weekends": "Sleep hours on weekends",
    "dpq040___feeling_tired_or_having_little_energy": "Feeling tired / little energy",
    "bpq020___ever_told_you_had_high_blood_pressure": "Ever told you had high blood pressure",
    "cdq010___shortness_of_breath_on_stairs/inclines": "Shortness of breath on stairs",
    "paq665___moderate_recreational_activities": "Moderate recreational activities",
    "alq111___ever_had_a_drink_of_any_kind_of_alcohol": "Ever had any alcohol",
    "mcq010___ever_been_told_you_have_asthma": "Ever told you had asthma",
}

VALUE_LABEL_MAP = {
    "gender": {
        "Male": "Male",
        "Female": "Female",
    },
    "bpq020___ever_told_you_had_high_blood_pressure": {
        "1.0": "Yes",
        "2.0": "No",
    },
    "cdq010___shortness_of_breath_on_stairs/inclines": {
        "1.0": "Yes",
        "2.0": "No",
    },
    "paq665___moderate_recreational_activities": {
        "1.0": "Yes",
        "2.0": "No",
    },
    "alq111___ever_had_a_drink_of_any_kind_of_alcohol": {
        "1.0": "Yes",
        "2.0": "No",
    },
    "mcq010___ever_been_told_you_have_asthma": {
        "1.0": "Yes",
        "2.0": "No",
    },
}

def prettify_transformed_feature_name(name: str) -> str:
    if name.startswith("num__missingindicator_"):
        raw = name.replace("num__missingindicator_", "")
        pretty_raw = FEATURE_LABEL_MAP.get(raw, raw)
        return f"{pretty_raw} missing"

    if name.startswith("num__"):
        raw = name.replace("num__", "")
        return FEATURE_LABEL_MAP.get(raw, raw)

    if name.startswith("cat__"):
        raw = name.replace("cat__", "")
        if "_" in raw:
            feature_part, value_part = raw.rsplit("_", 1)
            pretty_feature = FEATURE_LABEL_MAP.get(feature_part, feature_part)
            pretty_value = VALUE_LABEL_MAP.get(feature_part, {}).get(value_part, value_part)
            return f"{pretty_feature}: {pretty_value}"
        return FEATURE_LABEL_MAP.get(raw, raw)

    return name

best_estimator = selected_result["best_estimator"]
preprocessor = best_estimator.named_steps["preprocessor"]
model = best_estimator.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()

coef_df_raw = pd.DataFrame({
    "transformed_feature": feature_names,
    "coefficient": model.coef_[0],
})
coef_df_raw["abs_coefficient"] = coef_df_raw["coefficient"].abs()
coef_df_raw = coef_df_raw.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

coef_df_pretty = coef_df_raw.copy()
coef_df_pretty["pretty_feature"] = coef_df_pretty["transformed_feature"].apply(prettify_transformed_feature_name)
coef_df_pretty = coef_df_pretty[
    ["pretty_feature", "coefficient", "abs_coefficient", "transformed_feature"]
]

coef_df_nonzero = coef_df_pretty[coef_df_pretty["coefficient"] != 0].copy()
coef_df_nonzero = coef_df_nonzero.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

coef_df_nonzero.head(40)

## Save reports

In [ ]:
comparison_path = OUTPUT_DIR / "sleep_disorder_compact_model_comparison.csv"
threshold_path = OUTPUT_DIR / f"{selected_run}_threshold_scan.csv"
coef_path = OUTPUT_DIR / f"{selected_run}_coefficients_pretty.csv"

comparison_df.to_csv(comparison_path, index=False)
threshold_df.to_csv(threshold_path, index=False)
coef_df_pretty.to_csv(coef_path, index=False)

print("Saved:")
print(comparison_path.resolve())
print(threshold_path.resolve())
print(coef_path.resolve())

## Save model package and metadata

Update `threshold` after deciding on the operating point you want in product.

In [ ]:
disease = "sleep_disorder"
threshold = 0.50  # change if you choose a different operating point
threshold_label = str(threshold).replace(".", "")

feature_names = feature_sets[selected_run]
model_package = {
    "model": selected_result["best_estimator"],
    "threshold": threshold,
    "features": feature_names,
    "model_name": selected_run,
    "target": TARGET,
    "disease": disease,
}

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

joblib_path = MODELS_DIR / f"{disease}_{selected_run}_threshold_{threshold_label}.joblib"
joblib.dump(model_package, joblib_path)

print("Joblib saved to:", joblib_path.resolve())
print("Stored features:", feature_names)

In [ ]:
metadata = {
    "model_name": selected_run,
    "target": TARGET,
    "disease": disease,
    "threshold": threshold,
    "features": feature_names,
    "n_features": len(feature_names),
    "metrics": {
        "roc_auc": float(selected_result["roc_auc"]),
        "avg_precision": float(selected_result["avg_precision"]),
        "precision": float(selected_result["precision"]),
        "recall": float(selected_result["recall"]),
        "f1": float(selected_result["f1"]),
    },
    "best_params": selected_result["best_params"],
    "created_at": datetime.now(timezone.utc).isoformat(),
    "notes": "Compact sleep disorder model using quiz, demographics, medication count, and optional standard screening labs.",
}

metadata_path = MODELS_DIR / f"{disease}_{selected_run}_threshold_{threshold_label}.metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Metadata JSON saved to:", metadata_path.resolve())

In [ ]:
loaded_package = joblib.load(joblib_path)
with open(metadata_path, "r", encoding="utf-8") as f:
    loaded_metadata = json.load(f)

print("Loaded package keys:", loaded_package.keys())
print("Loaded model name:", loaded_package["model_name"])
print("Loaded threshold:", loaded_package["threshold"])
print("Loaded feature count:", len(loaded_package["features"]))
print("Loaded metadata disease:", loaded_metadata["disease"])